In [1]:
!nvidia-smi

Sat Mar 28 02:40:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Update the package list and install Nsight Compute
!apt-get update -yq
!apt-get install -yq nsight-compute

# 2. Add the CUDA binaries folder to your Python environment's PATH
import os
os.environ['PATH'] += ':/usr/local/cuda/bin'

# 3. Verify the installation
!ncu --version

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InReleas

In [3]:
!pip install torch torchvision torchsummary

## ResNet-50

### Training

In [4]:
!ncu --profile-from-start off \
--target-processes all \
-o resnet50_train_t4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py --dummy -a resnet50 --epochs 1 --profile-step 10 -b 8

!ncu -i resnet50_train_t4.ncu-rep --csv > resnet50_train_t4.csv

==PROF== Connected to process 3964 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'resnet50'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8      

### Evaluation

In [5]:
!ncu --profile-from-start off \
--target-processes all \
-o resnet50_eval_t4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py -e --dummy -a resnet50 --epochs 1 --profile-step 10 -b 8

!ncu -i resnet50_eval_t4.ncu-rep --csv > resnet50_eval_t4.csv

==PROF== Connected to process 7092 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'resnet50'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8      

## VGG-16

### Training

In [6]:
!ncu --profile-from-start off \
--target-processes all \
-o vgg16_train_t4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py --dummy -a vgg16 --epochs 1 --profile-step 10 -b 8

!ncu -i vgg16_train_t4.ncu-rep --csv > vgg16_train_t4.csv

==PROF== Connected to process 8202 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vgg16'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        [

### Evaluation

In [7]:
!nvidia-smi

Sat Mar 28 02:47:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             42W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!ncu --profile-from-start off \
--target-processes all \
-o vgg16_eval_t4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py -e --dummy -a vgg16 --epochs 1 --profile-step 10 -b 8

!ncu -i vgg16_eval_t4.ncu-rep --csv > vgg16_eval_t4.csv

==PROF== Connected to process 9569 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vgg16'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        [

## ViT-B-16

In [9]:
!pip install torchinfo -q
import torchvision.models as models
from torchinfo import summary

print("Generating Complexity Estimation for ViT-B-16...")
vit = models.vit_b_16()

# torchinfo correctly handles the ViT tuple outputs
summary(vit, input_size=(1, 3, 224, 224),
        col_names=["num_params", "mult_adds"],
        verbose=1)

Generating Complexity Estimation for ViT-B-16...
Layer (type:depth-idx)                        Param #                   Mult-Adds
VisionTransformer                             768                       --
├─Conv2d: 1-1                                 590,592                   115,756,032
├─Encoder: 1-2                                151,296                   --
│    └─Dropout: 2-1                           --                        --
│    └─Sequential: 2-2                        --                        --
│    │    └─EncoderBlock: 3-1                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-2                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-3                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-4                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-5                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-6                 7,087,872                 4,7

Layer (type:depth-idx)                        Param #                   Mult-Adds
VisionTransformer                             768                       --
├─Conv2d: 1-1                                 590,592                   115,756,032
├─Encoder: 1-2                                151,296                   --
│    └─Dropout: 2-1                           --                        --
│    └─Sequential: 2-2                        --                        --
│    │    └─EncoderBlock: 3-1                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-2                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-3                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-4                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-5                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-6                 7,087,872                 4,725,504
│    │    └─EncoderBlock: 3-7             

### Training

In [10]:
!ncu --profile-from-start off \
--target-processes all \
-o vit_train_t4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py --dummy -a vit_b_16 --epochs 1 --profile-step 10 -b 8

!ncu -i vit_train_t4.ncu-rep --csv > vit_train_t4.csv

==PROF== Connected to process 10121 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vit_b_16'
Input size: (3, 224, 224)
=> Skipping torchsummary for vit_b_16 to prevent hook corruption.
=> Dummy data is used!
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-pac

### Evaluation

In [11]:
!ncu --profile-from-start off \
--target-processes all \
-o vit_eval_t4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py -e --dummy -a vit_b_16 --epochs 1 --profile-step 10 -b 8

!ncu -i vit_eval_t4.ncu-rep --csv > vit_eval_t4.csv

==PROF== Connected to process 13622 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vit_b_16'
Input size: (3, 224, 224)
=> Skipping torchsummary for vit_b_16 to prevent hook corruption.
=> Dummy data is used!
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-pac